In [ ]:
# ============================================================
# One overlay image per patient from existing preprocessing result
# ------------------------------------------------------------
# 목적:
#   - 기존 전처리 결과(metadata_slices.csv) 사용
#   - 환자당 대표 slice 1장만 선택
#   - CT 위에 refined lung / pure lung overlay 그려서 저장
#
# 선택 기준:
#   - is_lung_range_slice == 1 우선
#   - 그 안에서 pure_lung_area가 가장 큰 slice 선택
#
# 색상:
#   - refined lung: cyan
#   - pure lung: yellow
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt


# ============================================================
# 1. CONFIG
# ============================================================

CFG = {
    # 기존 전처리 root
    # metadata_slices.csv가 있는 폴더
    "preprocess_root": r"E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1",

    # 결과 저장 폴더
    "out_dir_name": "one_overlay_per_patient",

    # group 필터
    "group_name": "Normal",

    # 특정 환자만 보고 싶으면 리스트에 넣기
    # 비워두면 전체 환자 대상
    "target_patient_ids": [
        # 예시
        # "subset6_1.3.6.1.4.1.14519.5.2.1.6279.6001.290410217650314119074833254861",
        # "subset7_1.3.6.1.4.1.14519.5.2.1.6279.6001.170825539570536865106681134236",
    ],

    # 저장 후 앞에서 몇 장 바로 화면에 보여줄지
    "preview_n": 5,

    # CT window
    "hu_min": -1000,
    "hu_max": 400,
}


PRE_ROOT = Path(CFG["preprocess_root"])
META_CSV = PRE_ROOT / "metadata_slices.csv"
OUT_DIR = PRE_ROOT / CFG["out_dir_name"]
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PRE_ROOT:", PRE_ROOT)
print("META_CSV:", META_CSV)
print("OUT_DIR:", OUT_DIR)


# ============================================================
# 2. 기본 함수
# ============================================================

def hu_to_uint8(arr, hu_min=-1000, hu_max=400):
    x = np.clip(arr.astype(np.float32), hu_min, hu_max)
    x = (x - hu_min) / float(hu_max - hu_min + 1e-8)
    x = np.clip(x * 255.0, 0, 255).astype(np.uint8)
    return x


def read_nii_array(path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    return img, arr


def choose_one_row_per_patient(patient_df):
    """
    대표 slice 1개 선택 규칙:
    1) is_lung_range_slice == 1 우선
    2) pure_lung_area가 가장 큰 slice
    3) pure_lung_area 없으면 refined_lung_area 기준
    """

    df = patient_df.copy()

    if "is_lung_range_slice" in df.columns:
        lung_df = df[df["is_lung_range_slice"] == 1].copy()
        if len(lung_df) > 0:
            df = lung_df

    if "pure_lung_area" in df.columns:
        idx = df["pure_lung_area"].astype(float).idxmax()
        return df.loc[idx]

    if "refined_lung_area" in df.columns:
        idx = df["refined_lung_area"].astype(float).idxmax()
        return df.loc[idx]

    # 둘 다 없으면 가운데 행
    return df.iloc[len(df) // 2]


def safe_name(name: str) -> str:
    return (
        str(name)
        .replace("\\", "_")
        .replace("/", "_")
        .replace(":", "_")
        .replace("*", "_")
        .replace("?", "_")
        .replace('"', "_")
        .replace("<", "_")
        .replace(">", "_")
        .replace("|", "_")
        .replace(" ", "_")
    )


# ============================================================
# 3. metadata 로드
# ============================================================

if not META_CSV.exists():
    raise FileNotFoundError(f"metadata_slices.csv 없음: {META_CSV}")

meta_df = pd.read_csv(META_CSV)

print("rows:", len(meta_df))
print("columns:", list(meta_df.columns))

if "group" in meta_df.columns:
    meta_df = meta_df[meta_df["group"].astype(str) == str(CFG["group_name"])].copy()

target_patient_ids = CFG["target_patient_ids"]

if len(target_patient_ids) > 0:
    meta_df = meta_df[meta_df["patient_id"].astype(str).isin(target_patient_ids)].copy()

if len(meta_df) == 0:
    raise RuntimeError("조건에 맞는 metadata row가 없음.")

print("filtered rows:", len(meta_df))
print("patient count:", meta_df["patient_id"].nunique())


# ============================================================
# 4. 환자별 대표 slice 선택
# ============================================================

selected_rows = []

for patient_id, patient_df in meta_df.groupby("patient_id"):
    row = choose_one_row_per_patient(patient_df)
    selected_rows.append(row)

selected_df = pd.DataFrame(selected_rows).reset_index(drop=True)

print("\nselected patient count:", len(selected_df))
display(selected_df[[
    c for c in [
        "patient_id",
        "slice_index",
        "is_lung_range_slice",
        "pure_lung_area",
        "refined_lung_area",
        "ct_1mm_lps_nii",
        "lung_refined_1mm_nii",
        "pure_lung_1mm_nii",
    ]
    if c in selected_df.columns
]])


# ============================================================
# 5. 환자별 overlay 이미지 저장 - 메모리 절약 + 기존 파일 스킵 버전
# ------------------------------------------------------------
# 핵심 수정:
#   - 이미 만들어진 PNG는 스킵
#   - 전체 volume을 float32로 캐싱하지 않음
#   - 필요한 z slice 한 장만 읽어서 overlay 저장
# ============================================================

import gc

save_rows = []


def resolve_existing_path(row, key, fallback_path):
    """
    1순위: metadata_slices.csv 안의 경로
    2순위: PRE_ROOT 기준 fallback 경로
    """

    if key in row.index:
        value = row[key]

        if pd.notna(value) and str(value).strip() != "":
            csv_path = Path(str(value))

            if csv_path.exists():
                return csv_path

    fallback_path = Path(fallback_path)

    if fallback_path.exists():
        return fallback_path

    return None


def read_nii_slice_array(path: Path, z: int):
    """
    NIfTI에서 필요한 z slice 한 장만 읽음.
    전체 volume을 float32로 캐싱하지 않아서 메모리 사용량이 작음.
    """

    path = Path(path)

    reader = sitk.ImageFileReader()
    reader.SetFileName(str(path))
    reader.ReadImageInformation()

    size = list(reader.GetSize())  # x, y, z

    if len(size) < 3:
        raise RuntimeError(f"3D NIfTI가 아님: {path}, size={size}")

    if z < 0 or z >= size[2]:
        raise RuntimeError(f"z 범위 오류: z={z}, size={size}, path={path}")

    reader.SetExtractIndex([0, 0, int(z)])
    reader.SetExtractSize([int(size[0]), int(size[1]), 1])

    img_slice = reader.Execute()
    arr = sitk.GetArrayFromImage(img_slice)

    # shape: (1, H, W) -> (H, W)
    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]

    return arr


for i, row in selected_df.iterrows():
    patient_id = str(row["patient_id"])
    z = int(row["slice_index"])

    out_path = OUT_DIR / f"{safe_name(patient_id)}_z{z:04d}_overlay.png"

    # --------------------------------------------------------
    # 이미 만들어진 파일은 읽기 전에 바로 스킵
    # --------------------------------------------------------
    if out_path.exists():
        print(f"[SKIP EXISTING] {out_path}")

        save_rows.append({
            "patient_id": patient_id,
            "z": z,
            "status": "skipped_existing",
            "overlay_png": str(out_path),
        })

        continue

    fallback_ct_path = (
        PRE_ROOT
        / "ct_1mm_lps"
        / CFG["group_name"]
        / f"{patient_id}_1mm_lps.nii.gz"
    )

    fallback_lung_path = (
        PRE_ROOT
        / "lung_masks_refined_1mm"
        / CFG["group_name"]
        / patient_id
        / "lung_refined_1mm.nii.gz"
    )

    fallback_pure_path = (
        PRE_ROOT
        / "pure_lung_1mm"
        / CFG["group_name"]
        / patient_id
        / "pure_lung_1mm.nii.gz"
    )

    ct_path = resolve_existing_path(
        row=row,
        key="ct_1mm_lps_nii",
        fallback_path=fallback_ct_path,
    )

    lung_path = resolve_existing_path(
        row=row,
        key="lung_refined_1mm_nii",
        fallback_path=fallback_lung_path,
    )

    pure_path = resolve_existing_path(
        row=row,
        key="pure_lung_1mm_nii",
        fallback_path=fallback_pure_path,
    )

    if ct_path is None:
        print(f"[SKIP] CT 경로 못 찾음: {patient_id}")
        print("  fallback:", fallback_ct_path)

        save_rows.append({
            "patient_id": patient_id,
            "z": z,
            "status": "missing_ct",
            "overlay_png": "",
        })

        continue

    if lung_path is None:
        print(f"[SKIP] lung mask 경로 못 찾음: {patient_id}")
        print("  fallback:", fallback_lung_path)

        save_rows.append({
            "patient_id": patient_id,
            "z": z,
            "status": "missing_lung_mask",
            "overlay_png": "",
        })

        continue

    if pure_path is None:
        print(f"[SKIP] pure lung mask 경로 못 찾음: {patient_id}")
        print("  fallback:", fallback_pure_path)

        save_rows.append({
            "patient_id": patient_id,
            "z": z,
            "status": "missing_pure_mask",
            "overlay_png": "",
        })

        continue

    try:
        # ----------------------------------------------------
        # 필요한 slice 한 장만 읽음
        # ----------------------------------------------------
        ct_slice = read_nii_slice_array(ct_path, z)
        lung_slice = read_nii_slice_array(lung_path, z) > 0
        pure_slice = read_nii_slice_array(pure_path, z) > 0

        if ct_slice.shape != lung_slice.shape or ct_slice.shape != pure_slice.shape:
            print(f"[SKIP] shape 불일치: {patient_id}")
            print("  ct:", ct_slice.shape)
            print("  lung:", lung_slice.shape)
            print("  pure:", pure_slice.shape)

            save_rows.append({
                "patient_id": patient_id,
                "z": z,
                "status": "shape_mismatch",
                "overlay_png": "",
            })

            continue

        ct_img = hu_to_uint8(
            ct_slice,
            hu_min=CFG["hu_min"],
            hu_max=CFG["hu_max"],
        )

        refined_area = int(lung_slice.sum())
        pure_area = int(pure_slice.sum())

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(ct_img, cmap="gray")

        # refined lung contour
        if refined_area > 0:
            ax.contour(
                lung_slice.astype(np.uint8),
                levels=[0.5],
                colors=["cyan"],
                linewidths=1.2,
            )

        # pure lung contour
        if pure_area > 0:
            ax.contour(
                pure_slice.astype(np.uint8),
                levels=[0.5],
                colors=["yellow"],
                linewidths=1.4,
            )

        ax.set_title(
            f"{patient_id}\n"
            f"z={z} | refined={refined_area} | pure={pure_area}",
            fontsize=10,
        )
        ax.axis("off")

        plt.tight_layout()
        plt.savefig(out_path, dpi=180, bbox_inches="tight")
        plt.close(fig)

        save_rows.append({
            "patient_id": patient_id,
            "z": z,
            "status": "saved",
            "refined_area": refined_area,
            "pure_area": pure_area,
            "overlay_png": str(out_path),
            "ct_1mm_lps_nii": str(ct_path),
            "lung_refined_1mm_nii": str(lung_path),
            "pure_lung_1mm_nii": str(pure_path),
        })

        print(f"[SAVED] {out_path}")

    except Exception as e:
        print(f"[ERROR] {patient_id}, z={z}")
        print("  error:", str(e))

        save_rows.append({
            "patient_id": patient_id,
            "z": z,
            "status": "error",
            "error": str(e),
            "overlay_png": "",
        })

    finally:
        # ----------------------------------------------------
        # 환자 하나 처리 후 메모리 정리
        # ----------------------------------------------------
        for name in [
            "ct_slice",
            "lung_slice",
            "pure_slice",
            "ct_img",
            "fig",
            "ax",
        ]:
            if name in locals():
                del locals()[name]

        gc.collect()


# ============================================================
# 6. 저장 요약 CSV
# ============================================================

save_df = pd.DataFrame(save_rows)
summary_csv = OUT_DIR / "one_overlay_per_patient_summary.csv"
save_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print("\n========== FINISHED ==========")
print("saved:", int((save_df["status"] == "saved").sum()) if "status" in save_df.columns else 0)
print("skipped existing:", int((save_df["status"] == "skipped_existing").sum()) if "status" in save_df.columns else 0)
print("total rows:", len(save_df))
print("summary csv:", summary_csv)

if len(save_df) > 0:
    display(save_df.head(CFG["preview_n"]))
# ============================================================
# 6. 저장 요약 CSV
# ============================================================

save_df = pd.DataFrame(save_rows)
summary_csv = OUT_DIR / "one_overlay_per_patient_summary.csv"
save_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print("\n========== FINISHED ==========")
print("saved image count:", len(save_df))
print("summary csv:", summary_csv)

if len(save_df) > 0:
    display(save_df.head(CFG["preview_n"]))

PRE_ROOT: E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1
META_CSV: E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\metadata_slices.csv
OUT_DIR: E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\one_overlay_per_patient
rows: 113262
columns: ['group', 'patient_id', 'patient_dir', 'slice_index', 'slice_png', 'lung_png', 'organ_exclusion_png', 'pure_lung_png', 'overlay_png', 'refined_lung_area', 'organ_exclusion_area', 'pure_lung_area', 'is_lung_range_slice', 'lung_range_z_start', 'lung_range_z_end', 'lung_range_num_slices', 'lung_range_found', 'lung_range_reason', 'lung_range_min_pure_lung_area_ratio', 'lung_range_margin_slices', 'refined_lung_area_ratio', 'organ_exclusion_area_ratio', 'pure_lung_area_ratio', 'nearest_original_slice_index', 'nearest_original_z_physical', 'distance_to_original_slice_mm', 'is_original_slice', 'ct_native_lps_nii', 'ct_1mm_lps_nii', 'lung_refined_1mm_nii', 'organ_exclusion_1mm_nii', 'pure_lung_1

,patient_id,slice_index,is_lung_range_slice,pure_lung_area,refined_lung_area,ct_1mm_lps_nii,lung_refined_1mm_nii,pure_lung_1mm_nii
0,normal001,100,1,51613,51705,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...
1,normal002,150,1,52022,52079,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...
2,normal003,160,1,59581,59620,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...
3,normal004,142,1,58485,58959,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...
4,normal005,124,1,76099,76159,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...
...,...,...,...,...,...,...,...,...
357,subset9_1.3.6.1.4.1.14519.5.2.1.6279.6001.3400...,174,1,50246,50276,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...
358,subset9_1.3.6.1.4.1.14519.5.2.1.6279.6001.4364...,175,1,44943,45004,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...
359,subset9_1.3.6.1.4.1.14519.5.2.1.6279.6001.4724...,129,1,67909,67918,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...
360,subset9_1.3.6.1.4.1.14519.5.2.1.6279.6001.8552...,211,1,70901,71558,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_f...


[SKIP EXISTING] E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\one_overlay_per_patient\normal001_z0100_overlay.png
[SKIP EXISTING] E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\one_overlay_per_patient\normal002_z0150_overlay.png
[SKIP EXISTING] E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\one_overlay_per_patient\normal003_z0160_overlay.png
[SKIP EXISTING] E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\one_overlay_per_patient\normal004_z0142_overlay.png
[SKIP EXISTING] E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\one_overlay_per_patient\normal005_z0124_overlay.png
[SKIP EXISTING] E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\one_overlay_per_patient\normal006_z0065_overlay.png
[SKIP EXISTING] E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_final_all_nonfast_v1\one_overlay_per_patient\normal007_z0094_overlay.png
[SKIP EXISTING] E:\jyp\ct_data_2d_

KeyboardInterrupt: 